# Benchmark de pronostico (PostgreSQL)

Este notebook ejecuta el benchmark completo de pronostico con comparaciones, sensibilidad e interpretacion automatica.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

CWD = Path.cwd().resolve()
candidates = [
    CWD,
    CWD.parent,
    CWD / "03_modelado" / "proyecto_ml_experimentos",
]
ROOT = next((p for p in candidates if (p / "src" / "forecasting.py").exists()), None)
if ROOT is None:
    raise RuntimeError("No se encontro la raiz de proyecto_ml_experimentos (src/forecasting.py).")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for m in ["src", "src.datasets_postgres", "src.forecasting"]:
    if m in sys.modules:
        del sys.modules[m]

from src.datasets_postgres import load_forecasting_dataset
from src.forecasting import benchmark_forecasting, benchmark_forecasting_sensitivity


In [ ]:
df = load_forecasting_dataset()
display(df.head())

display(
    Markdown(
        f'''
### Calidad del dataset
- Filas: **{len(df):,}**
- Productos: **{df['producto'].nunique():,}**
- Periodos: **{df['periodo'].nunique():,}**
- Nulos en `periodo`: **{int(df['periodo'].isna().sum()):,}**
'''
    )
)


In [ ]:
res = benchmark_forecasting(df)
sens = benchmark_forecasting_sensitivity(df)

models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
charts_dir = ROOT.parents[1] / "05_evidencias" / "graficas"
charts_dir.mkdir(parents=True, exist_ok=True)
res.to_csv(models_dir / "benchmark_forecasting.csv", index=False)
sens.to_csv(models_dir / "benchmark_forecasting_sensitivity.csv", index=False)

display(res)
display(sens.head(12))


In [ ]:
test_table = res[res["split"] == "test"].sort_values("WAPE").reset_index(drop=True)
plot_df = test_table[["modelo", "WAPE"]].copy()

plt.figure(figsize=(10, 4))
plt.bar(plot_df["modelo"], plot_df["WAPE"])
plt.title("WAPE en test por modelo")
plt.ylabel("WAPE")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(charts_dir / "forecasting_wape_test_por_modelo.png", dpi=150)
plt.show()

top_sens = sens.sort_values("WAPE_val").head(6).copy()
melt = top_sens[["modelo", "WAPE_train", "WAPE_val", "WAPE_test"]].melt(
    id_vars=["modelo"],
    var_name="split",
    value_name="WAPE",
)

plt.figure(figsize=(10, 4))
for model_name, sub in melt.groupby("modelo"):
    plt.plot(sub["split"], sub["WAPE"], marker="o", label=model_name)
plt.title("Brechas train/validation/test (top sensibilidad)")
plt.ylabel("WAPE")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(charts_dir / "forecasting_brechas_train_val_test.png", dpi=150)
plt.show()

display(Markdown(f"Graficas guardadas en: `{charts_dir}`"))


In [ ]:
best_test = test_table.iloc[0]
baseline_test = test_table[test_table["modelo"] == "Baseline_Lag1"].iloc[0]
improvement = float(baseline_test["WAPE"] - best_test["WAPE"])

best_sens = sens.sort_values(["WAPE_val", "WAPE_test"]).iloc[0]
gap_val_test = float(best_sens["gap_wape_val_test"])

if improvement >= 0.05 and gap_val_test <= 0.06:
    semaforo = "VERDE"
    estado = "modelo robusto y mejora clara frente al baseline"
elif improvement > 0 and gap_val_test <= 0.10:
    semaforo = "AMARILLO"
    estado = "mejora valida, pero requiere monitoreo de brecha"
else:
    semaforo = "ROJO"
    estado = "sin mejora consistente o con riesgo alto de sobreajuste"

display(
    Markdown(
        f'''
## Interpretacion
- Mejor modelo en test: **{best_test['modelo']}**.
- `WAPE` test del mejor: **{best_test['WAPE']:.4f}**.
- `WAPE` baseline lag-1: **{baseline_test['WAPE']:.4f}**.
- Mejora absoluta vs baseline: **{improvement:.4f}**.
- Mejor configuracion de sensibilidad (por validacion): **{best_sens['modelo']}**.
- Brecha `WAPE train -> val` en la mejor configuracion: **{best_sens['gap_wape_train_val']:.4f}**.

Lectura recomendada:
1. Si la brecha train-val sube demasiado, ajustar complejidad.
2. Mantener decision final por metrica de **test**.
3. Reportar siempre comparacion contra baseline lag-1.

## Conclusion ejecutiva
- Semaforo: **{semaforo}**.
- Estado: **{estado}**.
'''
    )
)
